In [2]:
from __future__ import annotations
import re
from typing import Any, Dict, List, Optional, Tuple
import pandas as pd
import numpy as np

def stage_to_idx(task: str, stage: Any) -> Optional[int]:
    if pd.isna(stage):
        return None
 
    s = stage.strip().upper()
    if task.lower() == "t":
        m = re.match(r"^T\s*([1-4])", s)
        if not m:
            return None
        return int(m.group(1)) - 1
    elif task.lower() == "n":
        m = re.match(r"^N\s*([0-3])", s)
        if not m:
            return None
        return int(m.group(1))
    return None


def extract_ground_truth(df: pd.DataFrame, task: str) -> Optional[pd.Series]:
    if task == "t":
        return df["T14"].astype(int)
    elif task == "n":
        return df["N03"].astype(int)
    return None


def macro_prf(y_true: List[int], y_pred: List[int], num_classes: int) -> Tuple[float, float, float]:
    import numpy as np  # local
    K = num_classes
    tp = np.zeros(K, dtype=int)
    fp = np.zeros(K, dtype=int)
    fn = np.zeros(K, dtype=int)

    for yt, yp in zip(y_true, y_pred):
        if yp == yt:
            tp[yt] += 1
        else:
            fp[yp] += 1
            fn[yt] += 1

    def safe_div(a: float, b: float) -> float:
        return a / b if b > 0 else 0.0

    precs = []
    recs = []
    f1s = []
    for k in range(K):
        p = safe_div(tp[k], tp[k] + fp[k])
        r = safe_div(tp[k], tp[k] + fn[k])
        f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
        precs.append(p)
        recs.append(r)
        f1s.append(f1)

    return float(sum(precs)) / K, float(sum(recs)) / K, float(sum(f1s)) / K

def pred_contains_groundtruth(task: str, pred_text, y_true_idx: int) -> bool:
    """GT 토큰(Tk/Nk)이 예측 문자열 어디에든 '부분 일치'로 포함되어 있으면 True."""
    if pd.isna(pred_text):
        return False
    s = str(pred_text)

    if task.lower() == "t":
        k = y_true_idx + 1  # 0->T1, 1->T2, ...
        pat = re.compile(rf"T\s*{k}(?:\s*[A-D])?(?!\s*\d)", re.IGNORECASE)
    else:
        k = y_true_idx      # 0->N0, 1->N1, ...
        pat = re.compile(rf"N\s*{k}(?:\s*[A-D])?(?!\s*\d)", re.IGNORECASE)

    return bool(pat.search(s))



In [3]:
from pathlib import Path
import re
import pandas as pd

# "__kewltm__t__"처럼 이중 밑줄 구분자를 기준으로 방법론/태스크만 캡처
PAT = re.compile(r"__(?P<method>[a-z0-9]+)__(?P<task>[tn])__")

def tcga_from_filename(path: Path):
    m = PAT.search(path.name)  # 파일명 어디에 있어도 찾음
    if not m:
        raise ValueError(f"Unexpected filename: {path.name}")
    return m.group("method"), m.group("task")

per_cancer_dir = Path("/home/yl3427/cylab/selfCorrectionAgent/runs")
for path in sorted(per_cancer_dir.glob("*.csv")):
    if path.name.startswith("manifest"):
        continue  
    df = pd.read_csv(path)
    cancer_type = path.stem.split("_")[0]
    method, task = tcga_from_filename(path)
    print(f"cancer_type: {cancer_type}, method: {method}, task: {task}")
    if method == "kewltm":
        df = df[~df["is_train"]]
    if task == "t":
        true_col = df["T14"]
    else:
        true_col = df["N03"]
    pred_col = df[f"{method}_stage"]
    pred_col = [stage_to_idx(task, stage) if stage else None for stage in pred_col]
    print(len(true_col), len([v for v in pred_col if v is not None]))


cancer_type: ACC, method: kewltm, task: n
84 25
cancer_type: ACC, method: kewltm, task: t
84 82
cancer_type: ACC, method: kewrag, task: n
88 48
cancer_type: ACC, method: kewrag, task: t
88 84
cancer_type: ACC, method: rag, task: n
88 40
cancer_type: ACC, method: rag, task: t
88 80
cancer_type: ACC, method: zscot, task: n
88 75
cancer_type: ACC, method: zscot, task: t
88 83
cancer_type: BLCA, method: kewltm, task: n
321 310
cancer_type: BLCA, method: kewltm, task: t
328 321
cancer_type: BLCA, method: kewrag, task: n
338 304
cancer_type: BLCA, method: kewrag, task: t
345 336
cancer_type: BLCA, method: rag, task: n
338 299
cancer_type: BLCA, method: rag, task: t
345 317
cancer_type: BLCA, method: zscot, task: n
338 315
cancer_type: BLCA, method: zscot, task: t
345 332
cancer_type: BRCA, method: kewltm, task: n
760 571
cancer_type: BRCA, method: kewltm, task: t
979 952
cancer_type: BRCA, method: kewrag, task: n
800 729
cancer_type: BRCA, method: kewrag, task: t
1031 983
cancer_type: BRCA, 

In [4]:
per_cancer_dir = Path("/home/yl3427/cylab/selfCorrectionAgent/runs2")
for path in sorted(per_cancer_dir.glob("*.csv")):
    if path.name.startswith("manifest"):
        continue  
    df = pd.read_csv(path)
    cancer_type = path.stem.split("_")[0]
    method, task = tcga_from_filename(path)
    print(f"cancer_type: {cancer_type}, method: {method}, task: {task}")
    if method == "kewltm":
        df = df[~df["is_train"]]
    if task == "t":
        true_col = df["T14"]
    else:
        true_col = df["N03"]
    pred_col = df[f"{method}_stage"]
    pred_col = [stage_to_idx(task, stage) if stage else None for stage in pred_col]
    print(len(true_col), len([v for v in pred_col if v is not None]))


cancer_type: ACC, method: kewltm, task: n
84 28


cancer_type: ACC, method: kewltm, task: t
84 84
cancer_type: ACC, method: kewrag, task: n
88 60
cancer_type: ACC, method: kewrag, task: t
88 85
cancer_type: ACC, method: rag, task: n
88 59
cancer_type: ACC, method: rag, task: t
88 85
cancer_type: ACC, method: zscot, task: n
88 80
cancer_type: ACC, method: zscot, task: t
88 86
cancer_type: BLCA, method: kewltm, task: n
321 320
cancer_type: BLCA, method: kewltm, task: t
328 326
cancer_type: BLCA, method: kewrag, task: n
338 314
cancer_type: BLCA, method: kewrag, task: t
345 342
cancer_type: BLCA, method: rag, task: n
338 327
cancer_type: BLCA, method: rag, task: t
345 332
cancer_type: BLCA, method: zscot, task: n
338 329
cancer_type: BLCA, method: zscot, task: t
345 341
cancer_type: BRCA, method: kewltm, task: n
760 681
cancer_type: BRCA, method: kewltm, task: t
979 964
cancer_type: BRCA, method: kewrag, task: n
800 772
cancer_type: BRCA, method: kewrag, task: t
1031 1013
cancer_type: BRCA, method: rag, task: n
800 768
cancer_type: BRCA,

In [5]:

per_cancer_dir = Path("/home/yl3427/cylab/selfCorrectionAgent/runs3")
for path in sorted(per_cancer_dir.glob("*.csv")):
    if path.name.startswith("manifest"):
        continue  
    df = pd.read_csv(path)
    cancer_type = path.stem.split("_")[0]
    method, task = tcga_from_filename(path)
    print(f"cancer_type: {cancer_type}, method: {method}, task: {task}")
    if method == "kewltm":
        df = df[~df["is_train"]]
    if task == "t":
        true_col = df["T14"]
    else:
        true_col = df["N03"]
    pred_col = df[f"{method}_stage"]
    pred_col = [stage_to_idx(task, stage) if stage else None for stage in pred_col]
    print(len([v for v in true_col if v is not None]), len([v for v in pred_col if v is not None]))

cancer_type: ACC, method: kewltm, task: n
84 32
cancer_type: ACC, method: kewltm, task: t
84 84
cancer_type: ACC, method: kewrag, task: n
88 66
cancer_type: ACC, method: kewrag, task: t
88 86
cancer_type: ACC, method: rag, task: n
88 67
cancer_type: ACC, method: rag, task: t
88 87
cancer_type: ACC, method: zscot, task: n
88 83
cancer_type: ACC, method: zscot, task: t
88 87
cancer_type: BLCA, method: kewltm, task: n
321 320
cancer_type: BLCA, method: kewltm, task: t
328 326
cancer_type: BLCA, method: kewrag, task: n
338 319
cancer_type: BLCA, method: kewrag, task: t
345 342
cancer_type: BLCA, method: rag, task: n
338 331
cancer_type: BLCA, method: rag, task: t
345 333
cancer_type: BLCA, method: zscot, task: n
338 335
cancer_type: BLCA, method: zscot, task: t
345 343
cancer_type: BRCA, method: kewltm, task: n
760 703
cancer_type: BRCA, method: kewltm, task: t
979 971
cancer_type: BRCA, method: kewrag, task: n
800 787
cancer_type: BRCA, method: kewrag, task: t
1031 1020
cancer_type: BRCA,

In [6]:
true_col

0     3.0
1     1.0
2     2.0
3     3.0
4     3.0
     ... 
60    2.0
61    3.0
62    2.0
63    2.0
64    3.0
Name: T14, Length: 65, dtype: float64

In [7]:
import pandas as pd
df = pd.read_csv("/home/yl3427/cylab/selfCorrectionAgent/runs2/CESC_T14N03__kewltm__t__mistralai_Mixtral-8x7B-Instruct-v0.1__seed42__20250908_051211.csv")
df[~df["is_train"]]

FileNotFoundError: [Errno 2] No such file or directory: '/home/yl3427/cylab/selfCorrectionAgent/runs2/CESC_T14N03__kewltm__t__mistralai_Mixtral-8x7B-Instruct-v0.1__seed42__20250908_051211.csv'

In [ ]:
print(df[df["kewrag_stage"].isnull()].kewrag_reasoning.values[0])

To determine the N stage of the Adrenocortical Carcinoma cancer patient, we will follow the AJCC Cancer Staging Manual (7th edition) N staging rules provided.

1. Review the pathology report to locate information about regional lymph nodes.
2. Assess if the regional lymph nodes can be assessed (pNX), if there is no regional lymph node metastasis (pN0), or if there is metastasis in regional lymph nodes (pN1, pN2, or pN3).

Step 1: Analyzing the pathology report
The pathology report includes information about the adrenal tumor, IVC invasive mass, and additional tumor from the crus of the right diaphragm. However, there is no explicit mention of regional lymph nodes or their metastasis.

Step 2: Staging the patient
Since the pathology report does not provide enough information about the regional lymph nodes, we cannot determine whether the lymph nodes were assessed (pNX), had no metastasis (pN0), or had metastasis (pN1, pN2, or pN3). Therefore, we cannot accurately assign an N stage based

In [ ]:
df

,type,case_submitter_id,patient_filename,ajcc_pathologic_t,T14,ajcc_pathologic_n,N03,text,kewrag_rules,kewrag_reasoning,kewrag_stage
0,ACC,TCGA-OR-A5J1,TCGA-OR-A5J1.8866FD87-4F6F-4D7E-B99A-7DD427ED3BB3,T2,1.0,N0,0.0,Deliver To: NOS P. Cuncertain inknown behavior...,['NX: Regional lymph nodes cannot be assessed'...,"To determine the N stage for this patient, we ...",N0
1,ACC,TCGA-OR-A5J2,TCGA-OR-A5J2.345C34EE-F031-41A4-B955-BA054F2A3CE1,T3,2.0,N0,0.0,Date of Birth: Sex: Female. Macroscopy. Two ve...,['NX: Regional lymph nodes cannot be assessed'...,1. The pathology report provides information a...,NX
2,ACC,TCGA-OR-A5J3,TCGA-OR-A5J3.83F53610-4F40-425B-9B69-BBB94234224F,T3,2.0,N0,0.0,Sex: Female. Macroscopy. One vessel containing...,['NX: Regional lymph nodes cannot be assessed'...,The pathology report provides information abou...,N0
3,ACC,TCGA-OR-A5J4,TCGA-OR-A5J4.39E300DC-C01F-46BC-AE4C-D834458F1FF7,T3,2.0,N1,1.0,Sex: Female. Macroscopy. One vessel containing...,['NX: Regional lymph nodes cannot be assessed'...,1. The pathology report does not mention any e...,N0
4,ACC,TCGA-OR-A5J5,TCGA-OR-A5J5.BB63299C-B7F7-4F08-998D-C30E56CD6065,T4,3.0,N0,0.0,Sex: Male. Macroscopy. One vessel containing a...,['NX: Regional lymph nodes cannot be assessed'...,To determine the N stage for this patient usin...,NX
5,ACC,TCGA-OR-A5J6,TCGA-OR-A5J6.F664D25E-9050-460D-AEAE-AA4BFA515453,T2,1.0,N0,0.0,Sex: Female. Macroscopy. Two vessel containing...,['NX: Regional lymph nodes cannot be assessed'...,To determine the N stage for this patient with...,NX
6,ACC,TCGA-OR-A5J7,TCGA-OR-A5J7.F9C1971C-5BE2-4722-9A6C-CF4A4AD04352,T3,2.0,N0,0.0,Sex: Female. Macroscopy. Two vessels containin...,['NX: Regional lymph nodes cannot be assessed'...,The pathology report provides information abou...,NX
7,ACC,TCGA-OR-A5J8,TCGA-OR-A5J8.EA68CD7A-9308-40CD-BB52-CEB36334A146,T3,2.0,N0,0.0,Sex: Male. Macroscopy. One vessel containing a...,['NX: Regional lymph nodes cannot be assessed'...,To determine the N stage for this patient with...,NX
8,ACC,TCGA-OR-A5J9,TCGA-OR-A5J9.4C775CF4-39D7-4246-87A4-4659783A1C18,T2,1.0,N0,0.0,Sex: Female. Macroscopy. One vessel containing...,['NX: Regional lymph nodes cannot be assessed'...,In order to determine the N stage for this pat...,NX - Regional lymph nodes cannot be assessed
9,ACC,TCGA-OR-A5JA,TCGA-OR-A5JA.6A505565-429B-4D8A-A591-AAB54CF43B66,T4,3.0,N0,0.0,Requesting Doctor's information: HISTOPATHOLOG...,['NX: Regional lymph nodes cannot be assessed'...,To determine the N stage of the Adrenocortical...,NaN


In [ ]:
df2 = pd.read_csv("/home/yl3427/cylab/selfCorrectionAgent/runs2/ACC_T14N03__zscot__n__mistralai_Mixtral-8x7B-Instruct-v0.1__seed42__20250908_041406.csv")
print(df2.iloc[9].zscot_reasoning)

The N stage for Adrenocortical Carcinoma is determined by the presence and extent of regional lymph node metastasis. According to the AJCC Cancer Staging Manual (7th edition), N category is defined as follows:

NX: Regional lymph nodes cannot be assessed.
N0: No regional lymph node metastasis.
N1: Metastasis in regional lymph node(s).

In this pathology report, there is no mention of any lymph node evaluation or metastasis. The report describes the primary tumor in the right adrenal gland, an IVC invasive mass, and additional tumors in the crus of the right diaphragm, but there is no information about regional lymph nodes. Therefore,

The N stage for this Adrenocortical Carcinoma is NX, as the regional lymph nodes cannot be assessed from the provided information.

However, since NX is not a valid option in the provided choices, I would select N0, assuming that there is no evidence of regional lymph node metastasis based on the given report.


In [ ]:
from pathlib import Path  
import pandas as pd

csv_path = Path("/home/yl3427/cylab/selfCorrectionAgent/runs2").resolve().glob("*zscot*.csv")
for p in csv_path:
    print(p.name.split("_")[0], p.name.split("__")[2])
    df = pd.read_csv(p)
    print(len(df))
    print(df)

THCA n
10
KIRC n
6
KIRP t
10
COAD t
10
LIHC t
10
LUAD n
10
PAAD n
10
PAAD t
10
CESC n
10
LUSC n
10
PRAD n
8
BRCA t
10
ESCA t
10
ACC n
10
CHOL n
9
LUSC t
10
READ t
10
BRCA n
7
HNSC t
10
PRAD t
10
UVM t
10
TGCT n
10
SKCM t
10
THCA t
10
KICH n
7
ESCA n
10
STAD t
10
BLCA t
10
ACC t
10
UVM n
7
CESC t
9
MESO t
9
COAD n
10
SKCM n
10
LIHC n
2
READ n
10
LUAD t
10
KIRC t
10
BLCA n
9
KIRP n
1
KICH t
10
STAD n
7
TGCT t
10
MESO n
10
HNSC n
9
CHOL t
10


In [ ]:
df

,type,case_submitter_id,patient_filename,ajcc_pathologic_t,T14,ajcc_pathologic_n,N03,text,zscot_reasoning,zscot_stage
0,CHOL,TCGA-3X-AAV9,TCGA-3X-AAV9.3E7DD345-7A2C-4D7B-B5A2-2207949186CA,T1,0.0,N0,0.0,CONFIDENTIAL. Demographics (for. verification ...,The T stage for cholangiocarcinoma is determin...,T2
1,CHOL,TCGA-3X-AAVA,TCGA-3X-AAVA.4C9A220D-03E9-41CF-8FA9-4B7FD05A625A,T2b,1.0,NaN,NaN,CONFIDENTIAL. Demographics (for. verification ...,The T stage for cholangiocarcinoma is determin...,T2b
2,CHOL,TCGA-3X-AAVB,TCGA-3X-AAVB.BEA4C577-73ED-48F0-9417-AA8BD143B5C8,T3,2.0,N1,1.0,CONFIDENTIAL. Demographics (for. verification ...,The T stage for cholangiocarcinoma is determin...,T3
3,CHOL,TCGA-3X-AAVC,TCGA-3X-AAVC.88206A8B-257B-4C0F-B42C-905DEA9483FB,T1,0.0,N0,0.0,CONFIDENTIAL. Demographics (for. verification ...,The T stage for cholangiocarcinoma is based on...,T1b
4,CHOL,TCGA-4G-AAZF,TCGA-4G-AAZF.D67904F4-8D02-4D23-9390-C864A7B5344D,T3,2.0,N0,0.0,Anatomic site: intrahepatic. Histology diagnos...,To determine the T stage of cholangiocarcinoma...,T2
5,CHOL,TCGA-4G-AAZG,TCGA-4G-AAZG.4E32E573-17EA-49A4-A90F-A0830E3279C9,T3,2.0,N0,0.0,Anatomic site: intrahepatic. Histology diagnos...,To determine the T stage of cholangiocarcinoma...,T2
6,CHOL,TCGA-4G-AAZO,TCGA-4G-AAZO.C6A65949-C248-4729-B41F-9C65A46A7AF7,T2a,1.0,N0,0.0,Anatomic site: intrahepatic. Histology diagnos...,To determine the T stage of cholangiocarcinoma...,T2
7,CHOL,TCGA-4G-AAZR,TCGA-4G-AAZR.0411F033-20FA-4F8F-AD51-A4EDF3E758BE,T2a,1.0,N0,0.0,Anatomic site: perihilar. Histology diagnosis:...,The T stage for cholangiocarcinoma is determin...,T3
8,CHOL,TCGA-4G-AAZT,TCGA-4G-AAZT.C75C91FD-D4DE-4A58-A4C7-8F020AAC59EB,T1,0.0,N0,0.0,Anatomic site: intrahepatic. Histology diagnos...,To determine the T stage of cholangiocarcinoma...,T2
9,CHOL,TCGA-W5-AA2G,TCGA-W5-AA2G.1E7EDF0B-283F-44F9-A89A-83DB12A0D58A,T1,0.0,N0,0.0,"Female. Surgery Date: DIAGNOSIS: Liver, right ...",The T stage of cholangiocarcinoma is determine...,T2
